In [16]:
%load_ext autoreload
%autoreload 2

In [17]:
import sqlite3
import csv

### Setup

In [25]:
SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
SOURCE_DATA = "verb_obj_cases_strict.db"
UNRESOLVED_IDX = "unresolved_idx.db"
RESULT = "verb_obj_cases_filtered.db"

In [26]:
verbs_to_filter = []
with open("C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/liivas_verbs_obj_case_freqs.csv", "r", encoding="utf-8") as f:
    data = csv.DictReader(f)
    for row in data:
        if 'x' in row["ei_saa_olla_sihitist"]:
            verbs_to_filter.append(tuple([row["verb"], row["verb_compound"]]))

In [27]:
verbs_to_filter

[('näima', ''),
 ('hakkama', ''),
 ('nägema', 'välja'),
 ('nähtuma', 'ette'),
 ('nappima', ''),
 ('tulema', ''),
 ('peetuma', ''),
 ('jääma', ''),
 ('tappima', ''),
 ('jaguma', ''),
 ('kandideerima', ''),
 ('kuuluma', ''),
 ('asuma', ''),
 ('tunduma', ''),
 ('vajuma', ''),
 ('nõustuma', ''),
 ('sattuma', ''),
 ('muutuma', ''),
 ('hävima', ''),
 ('plahvatama', ''),
 ('õhkuma', ''),
 ('suhtuma', ''),
 ('pääsema', ''),
 ('sekkuma', ''),
 ('valmima', ''),
 ('surema', ''),
 ('toimuma', ''),
 ('akkama', ''),
 ('katkema', ''),
 ('näima', 'ette'),
 ('armuma', ''),
 ('pürgima', ''),
 ('lähtuma', ''),
 ('mõjuma', ''),
 ('tekkima', ''),
 ('uppuma', ''),
 ('kostuma', ''),
 ('paiskuma', ''),
 ('sobima', ''),
 ('kõlama', ''),
 ('mängitama', ''),
 ('valmistuma', ''),
 ('sarnanema', ''),
 ('kanduma', ''),
 ('viicima', ''),
 ('kumama', ''),
 ('jätkuma', ''),
 ('kalduma', ''),
 ('langema', ''),
 ('vaatama', 'vastu'),
 ('lakkama', ''),
 ('soostuma', ''),
 ('ilmuma', ''),
 ('kuulduma', ''),
 ('peetuma', '

In [28]:
len(verbs_to_filter)

93

In [29]:
# checking if verb+verb_compound is among verb constructions that can't have obj
def is_verb_to_filter(verb: str, verb_compound: str) -> bool:
    for el in verbs_to_filter:
        if el[0] == verb and el[1] == verb_compound:
            return True
    return False

In [30]:
# checking if POS tag is among POS tags that can appear in obj
def is_POS_to_filter(POS: str) -> str:
    POS_allowed = ["S", "H", "P", "Y", "N", "O", "A", "U", "C"]
    current_POS = POS.split("|")

    has_POS_allowed = False

    for pos in current_POS:
        if pos in POS_allowed:
            has_POS_allowed = True
            break
    
    if has_POS_allowed:
        return "NO"
    else:
        return "YES"

- Kui verbil ainult üks peaargument, siis see on alus.
- Olema-verbi konstruktsioonides ei saa olla sihitist.
- Välja nägema ei saa olla sihitist. 
Põhjus:
-- kirjeldab seisundit
-- Indikatiivne tunnus: POS A|C|U, case NOM domineerivalt
-- Hüpotees: kirjeldab seiundit on KÕIK v MITTE MIDAGI tunnus
-- Hüpotees: peab olema compound
- Käskivas kõneviisis (imperatiiv) saab olla nominatiivis sihitis.
-- Hüpotees: iga verb kas nom või part, aga mitte korraga
Ehk visata välja juhud, kus sihitis on nominatiivis, aga pole käskivat kõneviisi.
- Sihitis alati käändsõna
-- sagedaseim S/H
-- asesõnad P
-- lühendid Y
-- arvsõna N, kuigi vähem selles süsteemis
-- omadussõna, kuigi vähem selles süsteemis
--- need tavalised elliptilistes lausetes

### I Olema-verbi konstruktsioonid

In [31]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.verb_olema
""")

cur.execute("""
CREATE TABLE result.verb_olema AS
SELECT
    *
FROM
    verb_strict_examples
WHERE
    verb='olema'
""")


cur.execute("""
DROP TABLE IF EXISTS idxs.unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.unresolved_idx
AS
SELECT
    tbl1.head_id AS idx
FROM
    verb_strict_examples AS tbl1
LEFT JOIN
    result.verb_olema AS tbl2
ON 
    tbl1.head_id=tbl2.head_id
WHERE 
    tbl2.head_id IS NULL
""")


con.close()

### II välja nägema

In [32]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.valja_nagema
""")

cur.execute("""
CREATE TABLE result.valja_nagema AS
SELECT
    *
FROM
    verb_strict_examples
INNER JOIN
    idxs.unresolved_idx AS tmp
ON
    verb_strict_examples.head_id=tmp.idx
WHERE
    verb='nägema'
AND
    verb_compound='välja'
""")

cur.execute("""
DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.new_unresolved_idx
AS
SELECT
    tbl1.idx
FROM
    idxs.unresolved_idx AS tbl1
LEFT JOIN
    result.valja_nagema AS tbl2
ON 
    tbl1.idx=tbl2.head_id
WHERE 
    tbl2.head_id IS NULL
""")

cur.execute("""
DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

### III muud verbikonstruktsioonid, millel ei saa olla obj

In [33]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA}")

con.create_function("is_verb_to_filter", 2, is_verb_to_filter)

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.verbs_without_obj
""")

cur.execute("""
CREATE TABLE result.verbs_without_obj AS
SELECT
    *
FROM
    verb_strict_examples
INNER JOIN
    idxs.unresolved_idx AS tmp
ON
    verb_strict_examples.head_id=tmp.idx
WHERE
    is_verb_to_filter(verb, verb_compound)
""")

cur.execute("""
DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.new_unresolved_idx
AS
SELECT
    tbl1.idx
FROM
    idxs.unresolved_idx AS tbl1
LEFT JOIN
    result.verbs_without_obj AS tbl2
ON 
    tbl1.idx=tbl2.head_id
WHERE 
    tbl2.head_id IS NULL
""")

cur.execute("""
DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

### IV obj kääne nom, aga verb pole imperatiivis

In [34]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.nom_not_imper
""")

cur.execute("""
CREATE TABLE result.nom_not_imper AS
SELECT
    *
FROM
    verb_strict_examples
INNER JOIN
    idxs.unresolved_idx AS tmp
ON
    verb_strict_examples.head_id=tmp.idx
WHERE
    current_case='nom'
AND
    verb_mood!='imper'
AND
    current_case_number!='pl'
""")

cur.execute("""
DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.new_unresolved_idx
AS
SELECT
    tbl1.idx
FROM
    idxs.unresolved_idx AS tbl1
LEFT JOIN
    result.nom_not_imper AS tbl2
ON 
    tbl1.idx=tbl2.head_id
WHERE 
    tbl2.head_id IS NULL
""")

cur.execute("""
DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

### Juhtumid, kus obj sõnaliik ei sobi

In [35]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA}")

con.create_function("is_POS_to_filter", 1, is_POS_to_filter)

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.wrong_POS
""")

cur.execute("""
CREATE TABLE result.wrong_POS AS
SELECT
    *
FROM
    verb_strict_examples
INNER JOIN
    idxs.unresolved_idx AS tmp
ON
    verb_strict_examples.head_id=tmp.idx
WHERE
    is_POS_to_filter(POS)='YES'
""")

cur.execute("""
DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.new_unresolved_idx
AS
SELECT
    tbl1.idx
FROM
    idxs.unresolved_idx AS tbl1
LEFT JOIN
    result.wrong_POS AS tbl2
ON 
    tbl1.idx=tbl2.head_id
WHERE 
    tbl2.head_id IS NULL
""")

cur.execute("""
DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

### Puhtad

In [36]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.cleaned
""")

cur.execute("""
CREATE TABLE result.cleaned AS
SELECT
    *
FROM
    verb_strict_examples
INNER JOIN
    idxs.unresolved_idx AS tmp
ON
    verb_strict_examples.head_id=tmp.idx
""")

cur.execute("""
DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.new_unresolved_idx
AS
SELECT
    tbl1.idx
FROM
    idxs.unresolved_idx AS tbl1
LEFT JOIN
    result.cleaned AS tbl2
ON 
    tbl1.idx=tbl2.head_id
WHERE 
    tbl2.head_id IS NULL
""")

cur.execute("""
DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

In [ ]:
# dropi read, kus kääne on vale?